# Remittance Cost Tracker: what it costs to send $200 home

Data: World Bank **Remittance Prices Worldwide (RPW)**, Q2 2016 to Q3 2025.
Every row is one service (a bank, a money transfer operator, a mobile operator) quoting the cost of sending **$200** on one route (corridor), in one quarter.

**How to run:** Runtime → *Restart session and run all*. Steps that are already done (download, Parquet) skip themselves.

### 1. Connect Google Drive
Everything is saved in one Drive folder, so the work survives when Colab restarts.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

### 2. Make the project folders
`exist_ok=True` means: create it if missing, do nothing if it is already there.

In [ ]:
import os
DATA = "/content/drive/MyDrive/remittance"   # project folder
OUT = f"{DATA}/outputs"                       # results go here
os.makedirs(DATA, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print("Project folder:", DATA)
print(sorted(os.listdir(DATA)))

### 3. Install and import the tools

In [ ]:
!pip install -q duckdb openpyxl pyarrow
import pandas as pd
import duckdb

### 4. Download the World Bank file (only once)
If the file is already in Drive and bigger than 1 MB, we skip the download.

In [ ]:
URL = "https://datacatalogfiles.worldbank.org/ddh-published/0037898/DR0095523/rpw_dataset_2011_2025_q3.xlsx"
XLSX = f"{DATA}/rpw_dataset_2011_2025_q3.xlsx"

if os.path.exists(XLSX) and os.path.getsize(XLSX) > 1_000_000:
    print("Already downloaded - skipping")
else:
    !wget -q -O "{XLSX}" "{URL}"
    print("Downloaded")
print(round(os.path.getsize(XLSX) / 1_000_000, 1), "MB")

### 5. Save the main sheet as Parquet (only once)
Reading Excel is slow; Parquet loads in seconds. Everything is saved as text first, we convert numbers ourselves later.

In [ ]:
MAIN = f"{DATA}/sheet_dataset_(from_q2_2016).parquet"

if os.path.exists(MAIN):
    print("Parquet already there - skipping the slow Excel step")
else:
    data = pd.read_excel(XLSX, sheet_name="Dataset (from Q2 2016)")
    data.astype(str).to_parquet(MAIN)
    print("Saved", data.shape)

### 6. Load the data

In [ ]:
data = pd.read_parquet(MAIN)
print(data.shape)   # rows, columns

### 7. Keep only the 14 columns we need, with simple names
- **cost_pct** = fee + exchange-rate margin, as a % of $200 (World Bank column `cc1 total cost %`)
- **fx_margin_pct** = the hidden part of the cost, inside the exchange rate

In [ ]:
keep = {
    "period": "period", "corridor": "corridor",
    "source_name": "send_country", "destination_name": "receive_country",
    "destination_region": "receive_region", "firm": "firm", "firm_type": "firm_type",
    "payment instrument": "pay_with", "access point": "send_via", "pickup method": "receive_as",
    "speed actual": "speed", "transparent": "transparent",
    "cc1 fx margin": "fx_margin_pct", "cc1 total cost %": "cost_pct",
}

data = data[list(keep)].rename(columns=keep)
data.head()

### 8. Turn the cost columns from text into numbers
`errors="coerce"` turns anything that is not a number (like `..`) into a blank (NaN).

In [ ]:
data["cost_pct"] = pd.to_numeric(data["cost_pct"], errors="coerce")
data["fx_margin_pct"] = pd.to_numeric(data["fx_margin_pct"], errors="coerce")
data[["cost_pct", "fx_margin_pct"]].describe().round(2)

### 9. Label blanks in the region column
The World Bank writes `..` when a value is missing. We call it *Unknown* so it reads clearly.

In [ ]:
data["receive_region"] = data["receive_region"].replace({"..": "Unknown", "nan": "Unknown"})
data["receive_region"].value_counts()

### 10. Split "2016_2Q" into year and quarter
`period_order` (e.g. 20162) lets us sort quarters correctly.

In [ ]:
data["year"] = data["period"].str[:4].astype(int)                                    # "2016_2Q" -> 2016
data["quarter"] = data["period"].str[5].astype(int)                                  # "2016_2Q" -> 2
data["period_label"] = data["year"].astype(str) + " Q" + data["quarter"].astype(str) # "2016 Q2"
data["period_order"] = data["year"] * 10 + data["quarter"]                           # 20162
data[["period", "period_label", "period_order"]].drop_duplicates().tail()

### 11. Check for problem values
Blank costs, negative costs (possible when the exchange rate is better than the market rate) and costs above 50% (likely errors).

In [ ]:
missing = data["cost_pct"].isna().sum()
negative = (data["cost_pct"] < 0).sum()
too_high = (data["cost_pct"] > 50).sum()
print("Missing:", missing, "| Negative:", negative, "| Above 50%:", too_high)

### 12. Remove impossible costs, but keep a full copy
`data_all` is the untouched copy, used in step 18 to test other cleaning rules.

In [ ]:
data_all = data.copy()

before = len(data)
data = data[data["cost_pct"].between(0, 50)]
print("Removed", before - len(data), "rows; kept", len(data))

### 13. How do people send and pay?

In [ ]:
print(data["send_via"].value_counts().head(15))
print()
print(data["pay_with"].value_counts().head(15))

### 14. Channel: fully digital, mixed, or in person only (our own definition)
`send_via` can list several access points, e.g. `Agent,Internet`.
- **Fully digital**: every access point is Internet or Mobile phone
- **Mixed**: some digital, some in person
- **In person only**: agent, bank branch, post office, ATM or call centre

In [ ]:
digital_points = {"Internet", "Mobile phone"}

def channel_of(send_via):
    if pd.isna(send_via):
        return "Unknown"
    points = {p.strip() for p in send_via.split(",")}   # "Agent,Internet" -> {"Agent", "Internet"}
    if points <= digital_points:          # every way to send is digital
        return "Fully digital"
    if points & digital_points:           # some digital, some not
        return "Mixed"
    return "In person only"

data["channel"] = data["send_via"].apply(channel_of)
data["channel"].value_counts()

## SQL starts here
DuckDB runs SQL straight on our pandas table. From here on, `data` is never overwritten; each result gets its own name.

In [ ]:
rpw = data.copy()
con = duckdb.connect()
con.register("rpw", rpw)
print(len(rpw), "rows ready for SQL")

### 15. Which quarter is the latest?

In [ ]:
latest = con.execute("SELECT MAX(period_order) FROM rpw").fetchone()[0]
label = con.execute(f"SELECT ANY_VALUE(period_label) FROM rpw WHERE period_order = {latest}").fetchone()[0]
print("Latest quarter is:", label)

### 16. Global average cost, two ways
- **avg_all_services**: every service counts once
- **avg_of_corridors**: first average each route, then average the routes (every route counts once)

In [ ]:
global_avg = con.execute(f"""
    SELECT ROUND(AVG(cost_pct), 2) AS avg_all_services,
           (SELECT ROUND(AVG(corridor_avg), 2)
            FROM (SELECT AVG(cost_pct) AS corridor_avg
                  FROM rpw WHERE period_order = {latest}
                  GROUP BY corridor))     AS avg_of_corridors,
           COUNT(*)                 AS services,
           COUNT(DISTINCT corridor) AS corridors
    FROM rpw WHERE period_order = {latest}
""").df()
global_avg

### 17. Which cleaning rule matches the World Bank's published 6.36% (Q3 2025)?
We test three rules on the untouched copy, each averaged both ways.

In [ ]:
lq = data_all[data_all["period_order"] == latest]   # latest quarter, nothing removed

tests = {
    "A. keep everything":           lq.dropna(subset=["cost_pct"]),
    "B. 0 to 50 only (current)":    lq[lq["cost_pct"].between(0, 50)],
    "C. keep negatives, drop > 50": lq[lq["cost_pct"] <= 50],
}

rows = []
for name, df in tests.items():
    rows.append({
        "filter": name,
        "services": len(df),
        "avg_by_service": round(df["cost_pct"].mean(), 2),
        "avg_by_corridor": round(df.groupby("corridor")["cost_pct"].mean().mean(), 2),
    })
method_check = pd.DataFrame(rows)
method_check

### 18. Average cost by quarter (the trend)

In [ ]:
trend = con.execute("""
    SELECT period_label, period_order,
           ROUND(AVG(cost_pct), 2) AS avg_cost_pct,
           COUNT(*) AS services
    FROM rpw
    GROUP BY period_label, period_order
    ORDER BY period_order
""").df()
trend.tail(8)

### 19. Most and least expensive routes
Loss on $200 = cost % × 2. Only routes with 3+ services, so one odd quote can't decide the result. The UN target (SDG 10.c) is 3%.

In [ ]:
corridors = con.execute(f"""
    SELECT corridor, send_country, receive_country, receive_region,
           COUNT(*) AS services,
           ROUND(AVG(cost_pct), 2)     AS avg_cost_pct,
           ROUND(MIN(cost_pct), 2)     AS cheapest_pct,
           ROUND(AVG(cost_pct) - 3, 2) AS gap_to_3pct,
           ROUND(AVG(cost_pct) * 2, 2) AS avg_loss_usd_on_200,
           ROUND(MIN(cost_pct) * 2, 2) AS cheapest_loss_usd_on_200
    FROM rpw
    WHERE period_order = {latest}
    GROUP BY corridor, send_country, receive_country, receive_region
    HAVING COUNT(*) >= 3
    ORDER BY avg_cost_pct DESC
""").df()
print(len(corridors), "routes with 3+ services\n")
print("MOST EXPENSIVE:\n", corridors.head(10).to_string(), "\n")
print("CHEAPEST:\n", corridors.tail(10).sort_values("avg_cost_pct").to_string())

### 20. Cost by type of company

In [ ]:
by_firm_type = con.execute(f"""
    SELECT firm_type,
           COUNT(*) AS services,
           ROUND(AVG(cost_pct), 2)     AS avg_cost_pct,
           ROUND(AVG(cost_pct) * 2, 2) AS avg_loss_usd_on_200
    FROM rpw WHERE period_order = {latest}
    GROUP BY firm_type ORDER BY avg_cost_pct DESC
""").df()
by_firm_type

### 21. Cost by channel (digital vs in person)

In [ ]:
by_channel = con.execute(f"""
    SELECT channel,
           COUNT(*) AS services,
           ROUND(AVG(cost_pct), 2)     AS avg_cost_pct,
           ROUND(AVG(cost_pct) * 2, 2) AS avg_loss_usd_on_200
    FROM rpw WHERE period_order = {latest}
    GROUP BY channel ORDER BY avg_cost_pct DESC
""").df()
by_channel

### 22. Cost by region, and sending money to India
`yearly_saving_usd`: someone sending $200 every month who switches from the average service to the cheapest one.

In [ ]:
by_region = con.execute(f"""
    SELECT receive_region,
           COUNT(DISTINCT corridor) AS corridors,
           ROUND(AVG(cost_pct), 2)  AS avg_cost_pct
    FROM rpw WHERE period_order = {latest}
    GROUP BY receive_region ORDER BY avg_cost_pct DESC
""").df()

to_india = con.execute(f"""
    SELECT send_country,
           COUNT(*) AS services,
           ROUND(AVG(cost_pct), 2) AS avg_cost_pct,
           ROUND(MIN(cost_pct), 2) AS cheapest_pct,
           ROUND((AVG(cost_pct) - MIN(cost_pct)) * 2 * 12, 2) AS yearly_saving_usd
    FROM rpw
    WHERE period_order = {latest} AND receive_country = 'India'
    GROUP BY send_country ORDER BY avg_cost_pct DESC
""").df()

print(by_region.to_string(), "\n")
print(to_india.to_string())

### 23. Every service on every route in the latest quarter (for the website)
The dashboard needs the full list so a visitor can pick any route and see each company's price.

In [ ]:
services = con.execute(f"""
    SELECT corridor, send_country, receive_country, receive_region,
           firm, firm_type, channel, speed,
           ROUND(cost_pct, 2)      AS cost_pct,
           ROUND(fx_margin_pct, 2) AS fx_margin_pct
    FROM rpw WHERE period_order = {latest}
    ORDER BY corridor, cost_pct
""").df()
print(len(services), "services on", services["corridor"].nunique(), "routes")
services.head()

### 24. Save every result to Drive, then disconnect
This must always be the **last** cell: after it, Drive is disconnected until you run step 1 again.

In [ ]:
results = {
    "global_avg": global_avg, "method_check": method_check, "trend": trend,
    "corridors_latest": corridors, "by_firm_type": by_firm_type,
    "by_channel": by_channel, "by_region": by_region, "to_india": to_india,
    "corridor_services": services,
}
for name, table in results.items():
    table.to_csv(f"{OUT}/{name}.csv", index=False)
print("Saved in", OUT, ":", sorted(os.listdir(OUT)))

drive.flush_and_unmount()